### Collect Events
Used for mapping category i.e. sports, econ, politics

In [1]:
import requests
import pandas as pd
import time
from utils import clean_text_cols, clean_datetime_cols

url_type = 'events'
base_url = f"https://api.elections.kalshi.com/trade-api/v2/{url_type}"

all_events_data = []
cursor = None
max_pages = 10 

print(f"Fetching all {url_type} data from Kalshi...")

for page in range(max_pages):
    params = {"status": "open", "limit": 200, "cursor": cursor}
    response = requests.get(base_url, params=params)
    
    if response.status_code != 200:
        print(f"Error {response.status_code}: {response.text}")
        break
        
    data = response.json()
    batch = data.get(url_type, [])
    all_events_data.extend(batch)
    
    cursor = data.get('cursor')
    print(f"Page {page+1} fetched. Total events: {len(all_events_data)}")
    
    if not cursor:
        break
    time.sleep(0.5)

events_df = pd.DataFrame(all_events_data)

# Mapping & Filtering
event_col_map = {
    'series_ticker': 'series_id',
    'event_ticker': 'event_id',
    'category': 'tags',
    'title': 'event_title',
    'sub_title': 'event_sub_title'
}
events_df = events_df[[c for c in event_col_map.keys() if c in events_df.columns]].rename(columns=event_col_map)


Fetching all events data from Kalshi...
Page 1 fetched. Total events: 200
Page 2 fetched. Total events: 400
Page 3 fetched. Total events: 600
Page 4 fetched. Total events: 800
Page 5 fetched. Total events: 1000
Page 6 fetched. Total events: 1200
Page 7 fetched. Total events: 1400
Page 8 fetched. Total events: 1600
Page 9 fetched. Total events: 1800
Page 10 fetched. Total events: 2000


### Collect Markets
This is the pricing data

In [2]:
url_type = 'markets'
base_url = f"https://api.elections.kalshi.com/trade-api/v2/{url_type}"

all_market_data = []
cursor = None
max_pages = 5 

print("Fetching markets from Kalshi...")
for page in range(max_pages):
    params = {'mve_filter': 'exclude', 'status': 'open', 'limit': 1000, 'cursor': cursor}
    response = requests.get(base_url, params=params)
    
    if response.status_code != 200:
        print(f"Error {response.status_code}: {response.text}")
        break
        
    data = response.json()
    batch = data.get(url_type, [])
    all_market_data.extend(batch)
    
    cursor = data.get('cursor')
    print(f"Page {page+1} complete. Total records: {len(all_market_data)}")
    
    if not cursor:
        break
    time.sleep(0.5)

market_df = pd.DataFrame(all_market_data)
market_df['description'] = market_df['rules_primary'] + market_df['rules_secondary'] 

market_col_map = {
    'ticker': 'market_id',
    'event_ticker': 'event_id',
    'title': 'event_title',
    'subtitle': 'event_sub_title',
    'description': 'description',
    'yes_ask_dollars': 'yes_ask',
    'yes_bid_dollars': 'yes_bid',
    'no_ask_dollars': 'no_ask',
    'no_bid_dollars': 'no_bid',
    'close_time': 'close_time',
    'expiration_time': 'expiration'
}

market_df = market_df[[c for c in market_col_map.keys() if c in market_df.columns]].rename(columns=market_col_map)

Fetching markets from Kalshi...
Page 1 complete. Total records: 1000
Page 2 complete. Total records: 2000
Page 3 complete. Total records: 3000
Page 4 complete. Total records: 4000
Page 5 complete. Total records: 5000


### Backfill & Merge
Ensure our events, and market match

In [3]:
all_market_tickers = market_df['event_id'].unique()
known_event_tickers = events_df['event_id'].unique()
missing_tickers = [t for t in all_market_tickers if t not in known_event_tickers]

backfilled_events = []
if missing_tickers:
    print(f"Backfilling {len(missing_tickers)} missing events...")
    for ticker in missing_tickers:
        res = requests.get(f"https://api.elections.kalshi.com/trade-api/v2/events/{ticker}")
        if res.status_code == 200:
            backfilled_events.append(res.json().get('event', {}))
        time.sleep(0.05) 

if backfilled_events:
    new_events_df = pd.DataFrame(backfilled_events)
    new_events_df = new_events_df[[c for c in event_col_map.keys() if c in new_events_df.columns]].rename(columns=event_col_map)
    full_events_df = pd.concat([events_df, new_events_df], ignore_index=True)
else:
    full_events_df = events_df

cols_to_use = full_events_df.columns.difference(market_df.columns).tolist()
cols_to_use.append('event_id')

# 2. Merge using only the unique columns from the events dataframe
kalshi_master_df = pd.merge(
    market_df, 
    full_events_df[cols_to_use], 
    on='event_id', 
    how='inner'
)

Backfilling 514 missing events...


In [4]:
# Clean using utils
kalshi_master_df['platform'] = 'kalshi'

kalshi_master_df = clean_text_cols(kalshi_master_df)
kalshi_master_df = clean_datetime_cols(kalshi_master_df, date_cols=['close_time', 'expiration'])
kalshi_master_df.to_csv('data/markets/kalshi_markets.csv', index=False)

kalshi_events_df = clean_text_cols(full_events_df)
kalshi_events_df['platform'] = 'kalshi'
kalshi_events_df.to_csv('data/events/kalshi_events.csv', index=False)
kalshi_events_df

/Users/blakeuribe/Desktop/quasibets_testing/utils.py:67: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors='coerce').dt.strftime('%Y-%m-%d')
/Users/blakeuribe/Desktop/quasibets_testing/utils.py:67: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors='coerce').dt.strftime('%Y-%m-%d')


,series_id,event_id,tags,event_title,event_sub_title,platform
0,KXELONMARS,KXELONMARS-99,world,elon musk visit mars in his lifetime,before 2099,kalshi
1,KXNEWPOPE,KXNEWPOPE-70,elections,the next pope be,before 2070,kalshi
2,KXWARMING,KXWARMING-50,climate and weather,world pass 2 degrees celsius over pre industri...,before 2050,kalshi
3,KXMARSVRAIL,KXMARSVRAIL-50,science and technology,human land on mars before california starts hi...,before 2050,kalshi
4,KXERUPTSUPER,KXERUPTSUPER-0,climate and weather,supervolcano erupt before 2050,a supervolcano,kalshi
...,...,...,...,...,...,...
2509,KXFIGHTMENTION,KXFIGHTMENTION-26MAR28CHIPRI,mentions,announcers at chiesa vs price,chiesa vs price,kalshi
2510,KXFIGHTMENTION,KXFIGHTMENTION-26MAR28GRABAR,mentions,announcers at grasso vs barber,grasso vs barber,kalshi
2511,KXPGAH2H,KXPGAH2H-MAST26SSCHRMCI,sports,full tournament head to head scheffler vs mcilroy,scheffler vs mcilroy,kalshi
2512,KXFIGHTMENTION,KXFIGHTMENTION-26MAR28ADEPYF,mentions,announcers at adesanya vs pyfer,adesanya vs pyfer,kalshi


In [5]:
kalshi_check = kalshi_events_df[kalshi_events_df['event_title'].str.contains('tx', case=False, na=False)].reset_index(drop=True)
clean_text_cols(kalshi_check)


,series_id,event_id,tags,event_title,event_sub_title,platform
0,KXHOUSERACE,KXHOUSERACE-TX38-26,elections,which party will win the house race for tx 38,tx 38,kalshi
1,KXHOUSERACE,KXHOUSERACE-TX37-26,elections,which party will win the house race for tx 37,tx 37,kalshi
2,KXHOUSERACE,KXHOUSERACE-TX36-26,elections,which party will win the house race for tx 36,tx 36,kalshi
3,KXHOUSERACE,KXHOUSERACE-TX33-26,elections,which party will win the house race for tx 33,tx 33,kalshi
4,KXHOUSERACE,KXHOUSERACE-TX31-26,elections,which party will win the house race for tx 31,tx 31,kalshi
5,KXHOUSERACE,KXHOUSERACE-TX30-26,elections,which party will win the house race for tx 30,tx 30,kalshi
6,KXHOUSERACE,KXHOUSERACE-TX29-26,elections,which party will win the house race for tx 29,tx 29,kalshi
7,KXHOUSERACE,KXHOUSERACE-TX27-26,elections,which party will win the house race for tx 27,tx 27,kalshi
8,KXHOUSERACE,KXHOUSERACE-TX26-26,elections,which party will win the house race for tx 26,tx 26,kalshi
9,KXHOUSERACE,KXHOUSERACE-TX25-26,elections,which party will win the house race for tx 25,tx 25,kalshi
